# Librerias

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Carga de la bbdd y reducción

In [ ]:
#cargar csv
ruta = r""
df =  pd.read_csv("ruta")

# Nos quedamos con las variables que usaremos para segmentar
cols = ["age", "job", "marital", "education", "balance", "deposit", "housing", "loan"]

df_perfil = df[cols].copy()

C:\Users\misab\AppData\Local\Temp\ipykernel_5884\3743423449.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tablas = pd.read_sql("SHOW TABLES", connection).iloc[:, 0].tolist()
C:\Users\misab\AppData\Local\Temp\ipykernel_5884\3743423449.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfs[tabla] = pd.read_sql(f"SELECT * FROM {tabla}", connection)


dict_keys(['BANK_marketing'])


# Perfil del cliente

## 1. Crear clustering

In [ ]:
#Crear matriz X lista para hacer clustering:

#Convertir variables a 0 y 1
for col in ["deposit", "housing", "loan"]:
    df_perfil[col] = df_perfil[col].map({"yes": 1, "no": 0})

#Dividir categóricas y numéricas
cat_cols = ["job", "marital", "education"]
num_cols = ["age", "balance", "deposit", "housing", "loan"]

#One-hot encoding para categóricas
encoder = OneHotEncoder(drop="first", sparse_output=False)
X_cat = encoder.fit_transform(df_perfil[cat_cols])

#DataFrame con nombres de columnas
cat_feature_names = encoder.get_feature_names_out(cat_cols)
X_cat = pd.DataFrame(X_cat, columns=cat_feature_names, index=df_perfil.index)

#Unimos numéricas + categóricas codificadas
X_num = df_perfil[num_cols]
X = pd.concat([X_num, X_cat], axis=1)

In [ ]:
#Empezar clustering:

#Escalar variables para poder hacer k-means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Elegir número de clústers
inertias = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.plot(K_range, inertias, marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inertia")
plt.title("Método del codo")
plt.show()

In [ ]:
#Entrenar K-Means según resultado gráfico:

k_opt = 4
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
df_perfil["cluster"] = kmeans.fit_predict(X_scaled)

## 2. Perfil de cada cluster

In [ ]:
#Uso de productos por cluster:
cluster_products = df_perfil.groupby("cluster")[["deposit","housing","loan"]].mean()
print(cluster_products)

In [ ]:
#Edad media y balance medio por cluster:
cluster_num_profile = df_perfil.groupby("cluster")[["age","balance"]].mean()
print(cluster_num_profile)

In [ ]:
#Distribución de job, marital, education por cluster:
cluster_job = pd.crosstab(df_perfil["cluster"], df_perfil["job"], normalize="index")
cluster_marital = pd.crosstab(df_perfil["cluster"], df_perfil["marital"], normalize="index")
cluster_education = pd.crosstab(df_perfil["cluster"], df_perfil["education"], normalize="index")

## 3. Conclusiones